In [73]:
from db_cloud_func import *

In [74]:
df = fn_read_data_cloud("gold","bist_master_combined_indicators")

In [75]:
df

,EXCHANGE,SYMBOL,FRVP_INTERVAL,FRVP_PERIOD_TYPE,FRVP_HIGHEST_DATE,FRVP_HIGHEST_VALUE,FRVP_ROW_COUNT_AFTER_HIGHEST,FRVP_DAY_COUNT_AFTER_HIGHEST,FRVP_LATEST_CLOSE_VALUE,FRVP_POC,...,PVT_R4,PVT_R5,PVT_S1,PVT_S2,PVT_S3,PVT_S4,PVT_S5,PVT_STATUS,CREATED_AT,CREATED_DAY


In [ ]:
df = fn_read_data_cloud("gold","crypto_master_combined_indicators")

In [68]:
df = fn_read_data_cloud("logs","log_bist_master_combined_indicators")
# df = df[df["CREATED_DAY"] == "24-03-2026"].head(40)

In [28]:
def calc_poc_cluster_score(df):

    def cluster(group):
        n = group["FRVP_HIGHEST_DATE"].nunique()
        vals = group["FRVP_POC"].values

        if n == 1:
            return 3

        spread = (vals.max() - vals.min()) / vals.min()

        if n >= 3:
            if spread == 0:
                return 10
            elif spread <= 0.03:
                return 3 + (1 - spread / 0.03) * 7
            else:
                return 3

        if n == 2:
            if spread == 0:
                return 7
            elif spread <= 0.03:
                return 3 + (1 - spread / 0.03) * 4
            else:
                return 3

        return 0

    # 🔥 DOĞRU YÖNTEM
    cluster_scores = (
        df.groupby(["EXCHANGE", "SYMBOL"])
        .apply(cluster)
        .rename("cluster_score")
        .reset_index()
    )

    # merge ile geri bağla
    df = df.merge(cluster_scores, on=["EXCHANGE", "SYMBOL"], how="left")

    return df["cluster_score"]

In [33]:
import numpy as np
import pandas as pd

# =========================
# HELPERS
# =========================
def to_float(col):
    return (
        col.astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# =========================
# MFI
# =========================
def calc_mfi_new(df):
    return (
        np.where(df["MFI"] > df["MFI_YESTERDAY"], 25, 0) +
        np.where(df["MFI"] > df["MFI_12DAY_AVG"], 50, 0) +
        np.where(df["MFI_DIRECTION"] == "Upward", 25, 0)
    )

# =========================
# RSI
# =========================
def calc_rsi_new(df):
    rsi = to_float(df["RSI"])
    rsi_ma = pd.to_numeric(df["RSI_MA"], errors="coerce")

    score1 = np.where(rsi > rsi_ma, 70, 0)

    days = df["RSI_CROSS_DAYS_AGO"]
    score2 = np.where(
        (df["RSI_STATUS"] == 1) & (days <= 14),
        np.maximum(0, 15 - days),
        0
    )

    score3 = np.where((rsi >= 30) & (rsi <= 70), 15, 0)

    return np.clip(score1 + score2 + score3, 0, 100)

# =========================
# EMA
# =========================
def calc_ema_new(df):
    def ema_score(status, cross, days):
        return np.where(
            (status == 1) & (cross == 1),
            np.maximum(0, 3 - (days / 10)),
            0
        )

    total = (
        ema_score(df["EMA_STATUS_5_20"], df["EMA_CROSS_5_20"], df["EMA_DAYS_SINCE_CROSS_5_20"]) +
        ema_score(df["EMA_STATUS_3_20"], df["EMA_CROSS_3_20"], df["EMA_DAYS_SINCE_CROSS_3_20"]) +
        ema_score(df["EMA_STATUS_3_14"], df["EMA_CROSS_3_14"], df["EMA_DAYS_SINCE_CROSS_3_14"])
    )

    return np.clip((total / 9) * 100, 0, 100)

# =========================
# VOLUME
# =========================
def calc_volume_new(df):
    vol_last = to_float(df["VOL_LASTDAY"])
    vol_yest = to_float(df["VOL_YESTERDAY"])
    vol_5 = to_float(df["VOL_AVG_5DAY"])
    vol_10 = to_float(df["VOL_AVG_10DAY"])
    vol_20 = to_float(df["VOL_AVG_20DAY"])

    return (
        np.where(vol_last > vol_yest, 25, 0) +
        np.where(vol_last > vol_5, 25, 0) +
        np.where(vol_5 > vol_10, 25, 0) +
        np.where(vol_5 > vol_20, 25, 0)
    )

# =========================
# VWAP
# =========================
def calc_vwap_new(df):
    close = df["FRVP_LATEST_CLOSE_VALUE"]
    vwap = df["VWAP"]

    dist = np.abs(close - vwap) / vwap

    return np.clip(
        np.where(dist <= 0.05, 0,
                 np.where(dist >= 0.10, 100,
                          ((dist - 0.05) / 0.05) * 100)),
        0, 100
    )

# =========================
# POC FRVP
# =========================

def calc_poc_frvp_new(df):

    close = df["FRVP_LATEST_CLOSE_VALUE"]
    poc = df["FRVP_POC"]
    val = df["FRVP_VAL"]
    vah = df["FRVP_VAH"]

    # 1
    score1 = np.clip((1 - ((close - poc) / poc) / 0.20) * 5, 0, 5)

    # 2 (cluster FIXED)
    score2 = calc_poc_cluster_score(df)

    # 3 ✅ GREEN FIX
    score3 = np.where(df["BS_BAR_STATUS"] == "GREEN", 5, 0)

    # 4
    score4 = np.where(
        (df["BS_OPEN_PRICE"] > poc) &
        (df["BS_BAR_STATUS"] == "GREEN"),
        5,
        0
    )

    # 5
    score5 = np.clip((1 - np.abs(poc - val) / val / 0.20) * 5, 0, 5)

    # 6
    dist_vah = np.abs(poc - vah) / vah
    score6 = np.where(
        dist_vah >= 0.10, 5,
        np.where(
            dist_vah <= 0.05, 0,
            ((dist_vah - 0.05) / 0.05) * 5
        )
    )

    total = score1 + score2 + score3 + score4 + score5 + score6

    return np.clip((total / 35) * 100, 0, 100)

# =========================
# TRADE LEVELS
# =========================

def calculate_trade_levels(df):

    close = df["FRVP_LATEST_CLOSE_VALUE"]
    df["entry_price"] = close * 1.005

    pivot = df["PIVOT"] if "PIVOT" in df.columns else pd.Series(np.nan, index=df.index)
    r2 = df["R2"] if "R2" in df.columns else pd.Series(np.nan, index=df.index)

    def choose_target(row):

        c = row["FRVP_LATEST_CLOSE_VALUE"]

        if c < row["FRVP_POC"]:
            return row["VWAP"]

        elif c < row["VWAP"]:
            return row["VWAP"]

        elif c < row["FRVP_VAH"]:
            return row["FRVP_VAH"]

        elif pd.notna(pivot.loc[row.name]) and c < pivot.loc[row.name]:
            return pivot.loc[row.name]

        elif pd.notna(r2.loc[row.name]):
            return r2.loc[row.name]

        else:
            return row["FRVP_VAH"]

    df["target_price"] = df.apply(choose_target, axis=1)

    df["target_pct"] = ((df["target_price"] - df["entry_price"]) / df["entry_price"]) * 100
    df["risk_pct"] = ((df["entry_price"] - df["stop_loss"]) / df["entry_price"]) * 100
    df["rr_ratio"] = df["target_pct"] / df["risk_pct"]

    df["pivot_display"] = np.where(
        pivot.isna(),
        "N/A",
        pivot
    )

    return df

# =========================
# MASTER FUNCTION
# =========================

def calculate_scores(df):

    df["poc_frvp_status"] = calc_poc_frvp_new(df)
    df["vwap_status"] = calc_vwap_new(df)
    df["ema_status"] = calc_ema_new(df)
    df["rsi_status"] = calc_rsi_new(df)
    df["mfi_status"] = calc_mfi_new(df)
    df["vol_status"] = calc_volume_new(df)

    # =========================
    # MASTER SCORE
    # =========================
    df["master_score"] = (
        df["poc_frvp_status"] * 0.65 +
        df["vwap_status"] * 0.02 +
        df["ema_status"] * 0.09 +
        df["rsi_status"] * 0.07 +
        df["mfi_status"] * 0.07 +
        df["vol_status"] * 0.10
    )

    df["master_score"] = np.clip(df["master_score"], 0, 100)

    # =========================
    # WATCHLIST
    # =========================
    df["watchlist"] = np.where(
        df["master_score"] >= 70, "BUY",
        np.where(df["master_score"] >= 50, "WATCH", "AVOID")
    )
    df["entry_price"] = df["FRVP_LATEST_CLOSE_VALUE"] * 1.005
    

    # =========================
    # STOP LOSS (ENTRY’den sonra!)
    # =========================
    df["stop_loss"] = df["entry_price"] * 0.97

    # =========================
    # TRADE LEVELS (ENTRY burada oluşuyor)
    # =========================
    df = calculate_trade_levels(df)

    return df

In [69]:
scores = calculate_scores(df)

In [70]:
print(scores.to_clipboard(index=False))

None


In [35]:
scores

,EXCHANGE,SYMBOL,FRVP_INTERVAL,FRVP_PERIOD_TYPE,FRVP_HIGHEST_DATE,FRVP_HIGHEST_VALUE,FRVP_ROW_COUNT_AFTER_HIGHEST,FRVP_DAY_COUNT_AFTER_HIGHEST,FRVP_LATEST_CLOSE_VALUE,FRVP_POC,...,vol_status,master_score,watchlist,entry_price,target_price,target_pct,stop_loss,risk_pct,rr_ratio,pivot_display
0,BIST,DGATE,hourly,2year,2025-08-25 14:00:00,92.70,1542,156,73.05,71.20,...,0,58.372171,WATCH,73.41525,75.31,2.580867,71.212792,3.0,0.860289,N/A
1,BIST,KBORU,hourly,4months,2026-02-24 08:00:00,24.04,263,27,18.85,18.00,...,0,63.983326,WATCH,18.94425,19.76,4.306056,18.375923,3.0,1.435352,N/A
2,BIST,KBORU,hourly,6months,2026-02-24 08:00:00,24.04,263,27,18.85,18.00,...,0,63.983326,WATCH,18.94425,19.76,4.306056,18.375923,3.0,1.435352,N/A
3,BIST,KBORU,hourly,1year,2026-02-24 08:00:00,24.04,263,27,18.85,18.00,...,0,63.983326,WATCH,18.94425,19.76,4.306056,18.375923,3.0,1.435352,N/A
4,BIST,KBORU,hourly,2year,2026-02-24 08:00:00,24.04,263,27,18.85,18.00,...,0,63.983326,WATCH,18.94425,19.76,4.306056,18.375923,3.0,1.435352,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,BIST,YKBNK,hourly,2year,2026-02-27 09:00:00,44.50,232,24,34.62,34.04,...,25,46.182940,AVOID,34.79310,36.22,4.101101,33.749307,3.0,1.367034,N/A
600,BIST,YYLGD,hourly,4months,2026-02-16 09:00:00,14.61,322,33,10.88,11.31,...,0,43.988062,AVOID,10.93440,11.80,7.916301,10.606368,3.0,2.638767,N/A
601,BIST,YYLGD,hourly,6months,2026-02-16 09:00:00,14.61,322,33,10.88,11.31,...,0,43.988062,AVOID,10.93440,11.80,7.916301,10.606368,3.0,2.638767,N/A
602,BIST,YYLGD,hourly,1year,2026-02-16 09:00:00,14.61,322,33,10.88,11.31,...,0,43.988062,AVOID,10.93440,11.80,7.916301,10.606368,3.0,2.638767,N/A


In [51]:
df.head(5)


,EXCHANGE,SYMBOL,FRVP_INTERVAL,FRVP_PERIOD_TYPE,FRVP_HIGHEST_DATE,FRVP_HIGHEST_VALUE,FRVP_ROW_COUNT_AFTER_HIGHEST,FRVP_DAY_COUNT_AFTER_HIGHEST,FRVP_LATEST_CLOSE_VALUE,FRVP_POC,...,vol_status,master_score,watchlist,entry_price,stop_loss,target_price,target_pct,risk_pct,rr_ratio,pivot_display
0,BIST,A1YEN,hourly,1year,2025-08-26 09:00:00,37.94,1467,148,31.12,31.82,...,75,57.908976,WATCH,31.2756,30.337332,31.63,1.133152,3.0,0.377717,N/A
1,BIST,A1YEN,hourly,2year,2024-04-08 09:00:00,48.58,4858,489,31.12,28.10,...,75,74.115773,BUY,31.2756,30.337332,32.20,2.955659,3.0,0.985220,N/A
2,BIST,A1YEN,hourly,4months,2026-01-13 11:00:00,35.00,490,50,31.12,31.60,...,75,59.613089,WATCH,31.2756,30.337332,31.85,1.836575,3.0,0.612192,N/A
3,BIST,A1YEN,hourly,6months,2026-01-13 11:00:00,35.00,490,50,31.12,31.60,...,75,59.613089,WATCH,31.2756,30.337332,31.85,1.836575,3.0,0.612192,N/A
4,BIST,ADEL,hourly,1year,2026-01-16 08:00:00,42.00,463,47,34.00,34.04,...,75,59.113816,WATCH,34.1700,33.144900,36.34,6.350600,3.0,2.116867,N/A


# --- TRIAGE KISMI----

In [64]:
def calculate_forward_test(df_signal, df_hourly):

    df_hourly["TS"] = pd.to_datetime(df_hourly["TS"])
    df_hourly["date"] = df_hourly["TS"].dt.date
    df_hourly["hour"] = df_hourly["TS"].dt.hour

    results = []

    for _, row in df_signal.iterrows():

        symbol = row["SYMBOL"]
        created_day = pd.to_datetime(row["CREATED_DAY"], dayfirst=True).date()

        df_sym = df_hourly[df_hourly["SYMBOL"] == symbol].copy()

        if df_sym.empty:
            results.append({**row, "triage_entry_day": created_day})
            continue

        # =========================
        # VALID TRADE DAYS (08:00 varsa yeterli)
        # =========================
        valid_days = (
            df_sym.groupby("date")["hour"]
            .agg(list)
        )

        valid_days = valid_days[
            valid_days.apply(lambda h: 8 in h)
        ].index.tolist()

        future_days = [d for d in sorted(valid_days) if d > created_day]

        # =========================
        # T+1 kontrol
        # =========================
        if len(future_days) == 0:
            results.append({
                **row,
                "triage_entry_day": created_day,
                "day_after_11_chart": "N/A",
                "day_after_11_close": "N/A",
                "expected_entry_price": "N/A"
            })
            continue

        t1 = future_days[0]

        # =========================
        # 08:00 mum (ilk mum)
        # =========================
        candle_10 = df_sym[
            (df_sym["date"] == t1) &
            (df_sym["hour"] == 8)
        ]

        if candle_10.empty:
            day_after_chart = "N/A"
            day_after_close = np.nan
        else:
            c = candle_10.iloc[0]
            day_after_chart = "GREEN" if c["CLOSE"] > c["OPEN"] else "RED"
            day_after_close = c["CLOSE"]

        # =========================
        # ENTRY
        # =========================
        expected_entry = (
            day_after_close * 1.005
            if not np.isnan(day_after_close)
            else np.nan
        )

        # =========================
        # FORWARD DAYS (1-7)
        # =========================
        day_results = {}

        for i in range(1, 8):

            if len(future_days) < i:
                day_results[f"day+{i}_close"] = "N/A"
                day_results[f"day+{i}_close_%"] = "N/A"
                continue

            d = future_days[i-1]

            # 🔥 KAPANIŞ = GÜNÜN SON MUMU
            day_rows = df_sym[df_sym["date"] == d].sort_values("TS")

            if day_rows.empty:
                # zincir kır
                for j in range(i, 8):
                    day_results[f"day+{j}_close"] = "N/A"
                    day_results[f"day+{j}_close_%"] = "N/A"
                break

            day_close = day_rows.iloc[-1]["CLOSE"]

            if np.isnan(expected_entry):
                pct = "N/A"
            else:
                pct = ((day_close - expected_entry) / expected_entry) * 100

            day_results[f"day+{i}_close"] = day_close
            day_results[f"day+{i}_close_%"] = pct

        # =========================
        # FINAL
        # =========================
        results.append({
            **row,

            "triage_entry_day": created_day,
            "day_after_11_chart": day_after_chart,
            "day_after_11_close": day_after_close,
            "expected_entry_price": expected_entry,

            **day_results
        })

    return pd.DataFrame(results)

In [38]:
df_hourly = fn_read_data_cloud("raw","bist_hourly_archive")

In [71]:
forward = calculate_forward_test(df, df_hourly)


In [72]:
print(forward.to_clipboard(index=False))

None


In [59]:
created_day = pd.to_datetime("2026-03-24")

start_date = (created_day - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
end_date = (created_day + pd.Timedelta(days=12)).strftime("%Y-%m-%d")

In [60]:
query = f"""
SELECT "EXCHANGE","SYMBOL","TS","OPEN","CLOSE","VOLUME"
FROM raw.bist_hourly_archive
WHERE "SYMBOL" = 'A1YEN'
AND "TS" BETWEEN '{start_date}' AND '{end_date}'
ORDER BY "TS"
"""

In [ ]:
df_a1yen = pd.read_sql_query(query, engine)

In [63]:

print(df_a1yen.to_clipboard(index=False))

None


In [ ]:
# finish

In [ ]:
def build_dataset(df_signal):

    results = []

    for _, row in df_signal.iterrows():

        symbol = row["SYMBOL"]
        created_day = pd.to_datetime(row["CREATED_DAY"])

        start, end = get_date_range(created_day)

        df_hourly = fn_read_hourly_filtered(
            symbol,
            start,
            end
        )

        if df_hourly.empty:
            continue

        df_res = build_trading_dataset(
            pd.DataFrame([row]),
            df_hourly
        )

        if not df_res.empty:
            results.append(df_res)

    return pd.concat(results, ignore_index=True)

In [ ]:
from datetime import timedelta

def get_date_range(created_day):
    start = created_day - timedelta(days=40)
    end = created_day + timedelta(days=2)
    return start, end

In [ ]:
from db_cloud_func import read_sql_case_safe, engine
import pandas as pd
import numpy as np
from datetime import timedelta

# =========================
# DB DATA FETCH
# =========================
def fetch_hourly(symbol, start_date, end_date):
    query = f'''
        SELECT "EXCHANGE","SYMBOL","TS","OPEN","CLOSE","VOLUME"
        FROM "raw"."bist_hourly_archive"
        WHERE "SYMBOL" = '{symbol}'
        AND "TS" BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY "TS"
    '''
    return read_sql_case_safe(engine, query)

# =========================
# FIRST 2-HOUR CANDLE (08+09)
# =========================
def build_first_candle(df):
    df["TS"] = pd.to_datetime(df["TS"])
    df["date"] = df["TS"].dt.date
    df["hour"] = df["TS"].dt.hour

    df08 = df[df["hour"] == 8]
    df09 = df[df["hour"] == 9]

    merged = df08.merge(df09, on=["SYMBOL","date"], suffixes=("_08","_09"))

    merged["open"] = merged["OPEN_08"]
    merged["close"] = merged["CLOSE_09"]
    merged["volume"] = merged["VOLUME_08"] + merged["VOLUME_09"]

    return merged[["SYMBOL","date","open","close","volume"]].sort_values("date")

In [ ]:
def build_dataset(df_signal):

    results = []

    for _, row in df_signal.iterrows():

        symbol = row["SYMBOL"]
        exchange = row["EXCHANGE"]
        created_day = pd.to_datetime(row["CREATED_DAY"]).date()

        df_hourly = fetch_hourly(
            symbol,
            created_day - timedelta(days=40),
            created_day + timedelta(days=5)
        )

        if df_hourly.empty:
            continue

        df_first = build_first_candle(df_hourly)

        # =========================
        # T+1
        # =========================
        next_days = df_first[df_first["date"] > created_day]
        if next_days.empty:
            continue

        t1 = next_days.iloc[0]
        t1_day = t1["date"]

        # =========================
        # T
        # =========================
        hist = df_first[df_first["date"] <= created_day].sort_values("date").reset_index(drop=True)
        if hist.empty:
            continue

        t_row = hist.iloc[-1]

        # =========================
        # CONTACTLESS MOMENTUM (FINAL)
        # =========================
        prev_close = t_row["close"]
        t1_open = t1["open"]
        t1_close = t1["close"]

        contactless = (t1_open > prev_close) and (t1_close > t1_open)

        # =========================
        # VOLUME
        # =========================
        t1_volume = t1["volume"]
        t_volume = t_row["volume"]

        last3 = hist.iloc[-3:]["volume"]
        avg3 = last3.mean() if len(last3) == 3 else np.nan

        last15 = hist.iloc[-15:]["volume"]
        avg15 = last15.mean() if len(last15) == 15 else np.nan

        volume_inc_last3 = (
            not np.isnan(avg3) and
            t1_volume > avg3 and
            t1_volume > t_volume
        )

        volume_inc_last15 = (
            not np.isnan(avg15) and
            t1_volume > avg15 and
            t1_volume > t_volume
        )

        # =========================
        # BOT SIGNAL
        # =========================
        bot_signal = contactless and volume_inc_last3 and volume_inc_last15

        # =========================
        # POSITION
        # =========================
        position = "OPEN" if bot_signal else "NONE"

        # =========================
        # BUY PRICE (11:00 UTC)
        # =========================
        buy_price = np.nan

        if position == "OPEN":
            buy_candle = df_hourly[
                (pd.to_datetime(df_hourly["TS"]).dt.date == t1_day) &
                (pd.to_datetime(df_hourly["TS"]).dt.hour == 11)
            ]

            if not buy_candle.empty:
                buy_price = buy_candle.iloc[0]["OPEN"]

        # =========================
        # TARGET
        # =========================
        target_price = buy_price * 1.10 if not np.isnan(buy_price) else np.nan

        # =========================
        # LAST PRICE
        # =========================
        last_price = df_hourly.iloc[-1]["CLOSE"]

        # =========================
        # SELL LOGIC
        # =========================
        sell_price = np.nan

        if position == "OPEN":
            if last_price < row["stop_loss"]:
                sell_price = row["stop_loss"]
                position = "CLOSED"
            elif last_price > target_price:
                sell_price = target_price
                position = "CLOSED"

        # =========================
        # P&L
        # =========================
        pnl = sell_price / buy_price if position == "CLOSED" and not np.isnan(buy_price) else np.nan
        unrealized = last_price / buy_price if position == "OPEN" and not np.isnan(buy_price) else np.nan

        # =========================
        # DAY PERFORMANCE
        # =========================
        def get_day_return(offset):
            d = created_day + timedelta(days=offset)
            d_data = df_first[df_first["date"] == d]

            if d_data.empty:
                return np.nan

            return (d_data.iloc[0]["close"] / prev_close - 1) * 100

        results.append({
            "EXCHANGE": exchange,
            "SYMBOL": symbol,
            "triaje_enter": created_day,
            "triaje_score": row["master_score"],
            "poc_px": row["FRVP_POC"],
            "stop_loss": row["stop_loss"],
            "last_price": last_price,

            "triaje+1___check11_candle": t1_open,
            "contactless_momentum": contactless,
            "volume_inc_last3": volume_inc_last3,
            "volume_inc_last15": volume_inc_last15,
            "bot_signal": "BUY" if bot_signal else "",

            "position": position,
            "buy_price": buy_price,
            "sell_price": sell_price,
            "target_price": target_price,
            "P&L": pnl,
            "unrealized_P&L": unrealized,

            "Day_1": get_day_return(1),
            "Day_2": get_day_return(2),
            "Day_3": get_day_return(3),
            "Day_4": get_day_return(4),
            "Day_5": get_day_return(5),
        })

    return pd.DataFrame(results)

In [ ]:
df_result = build_dataset(scores)

In [ ]:
print(df_result.to_clipboard(index=False))